In [1]:
import cv2
import os

# -------- settings --------
samples_dir = "Samples"
output_dir = "patches_test"
patch_size = 256
stride = patch_size // 2
# --------------------------

# Create folders
for cls in ["0", "1"]:
    os.makedirs(os.path.join(output_dir, cls), exist_ok=True)

# Loop over images 001 to 005
image_names = [f"{i:03d}.jpg" for i in range(1, 2)]

try:
    for img_name in image_names:
        image_path = os.path.join(samples_dir, img_name)

        img = cv2.imread(image_path)
        if img is None:
            print(f"Could not load: {image_path}")
            continue

        h, w = img.shape[:2]
        print(f"\nProcessing {img_name} ({w} x {h})")

        for y in range(0, h - patch_size + 1, stride):
            for x in range(0, w - patch_size + 1, stride):

                patch = img[y:y + patch_size, x:x + patch_size].copy()

                cv2.imshow("Patch", patch)

                print(f"\n{img_name} - Patch (x={x}, y={y})")
                print("0 = no creature")
                print("1 = creature")
                print("s = skip | q = quit")

                key = cv2.waitKey(0) & 0xFF

                if key == ord('q'):
                    raise KeyboardInterrupt

                if key == ord('s'):
                    continue

                if key in [ord('0'), ord('1')]:
                    label = chr(key)

                    filename = f"{img_name}_x{x}_y{y}.jpg"
                    save_path = os.path.join(output_dir, label, filename)

                    cv2.imwrite(save_path, patch)
                    print(f"Saved: {save_path}")
                else:
                    print("Invalid key")

finally:
    cv2.destroyAllWindows()
    print("Finished")


Processing 001.jpg (1280 x 960)

001.jpg - Patch (x=0, y=0)
0 = no creature
1 = creature
s = skip | q = quit
Saved: patches_test/0/001.jpg_x0_y0.jpg

001.jpg - Patch (x=128, y=0)
0 = no creature
1 = creature
s = skip | q = quit
Saved: patches_test/0/001.jpg_x128_y0.jpg

001.jpg - Patch (x=256, y=0)
0 = no creature
1 = creature
s = skip | q = quit
Saved: patches_test/1/001.jpg_x256_y0.jpg

001.jpg - Patch (x=384, y=0)
0 = no creature
1 = creature
s = skip | q = quit
Saved: patches_test/1/001.jpg_x384_y0.jpg

001.jpg - Patch (x=512, y=0)
0 = no creature
1 = creature
s = skip | q = quit
Saved: patches_test/0/001.jpg_x512_y0.jpg

001.jpg - Patch (x=640, y=0)
0 = no creature
1 = creature
s = skip | q = quit
Saved: patches_test/0/001.jpg_x640_y0.jpg

001.jpg - Patch (x=768, y=0)
0 = no creature
1 = creature
s = skip | q = quit
Saved: patches_test/1/001.jpg_x768_y0.jpg

001.jpg - Patch (x=896, y=0)
0 = no creature
1 = creature
s = skip | q = quit
Saved: patches_test/1/001.jpg_x896_y0.jpg

00

In [3]:
import os
print(os.path.exists("samples"))


True


In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory

# -------- settings --------
data_dir = "patches_test"
img_size = (256, 256)
batch_size = 32
# --------------------------

# Load dataset from folder structure:
# patches_test/0/*.jpg
# patches_test/1/*.jpg
train_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

val_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds.class_names
print("Classes:", class_names)

# Make performance better
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Simple CNN
model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(256, 256, 3)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid")   # binary classification
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

model.save("creature_cnn.keras")
print("Model saved.")

Found 228 files belonging to 2 classes.
Using 183 files for training.
Found 228 files belonging to 2 classes.
Using 45 files for validation.
Classes: ['0', '1']
Epoch 1/10


/Users/betsabeescobedo/cnn_env/lib/python3.12/site-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 369ms/step - accuracy: 0.4590 - loss: 1.2918 - val_accuracy: 0.3556 - val_loss: 0.7243
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 360ms/step - accuracy: 0.6448 - loss: 0.6343 - val_accuracy: 0.7556 - val_loss: 0.5694
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 342ms/step - accuracy: 0.7158 - loss: 0.5711 - val_accuracy: 0.7111 - val_loss: 0.5402
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 342ms/step - accuracy: 0.8033 - loss: 0.4672 - val_accuracy: 0.8222 - val_loss: 0.4316
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 339ms/step - accuracy: 0.8087 - loss: 0.4576 - val_accuracy: 0.7778 - val_loss: 0.4435
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 341ms/step - accuracy: 0.8470 - loss: 0.4200 - val_accuracy: 0.8667 - val_loss: 0.4233
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 343ms/step - accuracy: 0.7978 - loss: 0.4487 - val_accuracy: 0.8667 - val_loss: 0.4159
Epoch 8/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 344ms/step - accuracy: 0.8579 - loss: 0.3903 - val_accuracy: 0.9111 - val_loss: 0.4492
Epo

In [9]:
import tensorflow as tf
import cv2
import numpy as np

model = tf.keras.models.load_model("creature_cnn.keras")

def predict_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (256, 256))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    prob = model.predict(img)[0][0]

    if prob >= 0.5:
        return "creature", float(prob)
    else:
        return "not creature", float(prob)

label, confidence = predict_image("samples/013.jpg")
print(label, confidence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
creature 0.9999869465827942
